In [74]:

#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126 --proxy http://rb-proxy-apac.bosch.com:8080

In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import Cityscapes
from torchvision.transforms import functional as TF
from torchvision.transforms import ColorJitter, InterpolationMode, RandomApply
from PIL import Image
from tqdm import tqdm


# Pre-Configuration

In [ ]:
# Dataset
DATA_ROOT = "./data"
CACHE_ROOT = "./cache"

NUM_CLASSES = 19
IGNORE_INDEX = 255

# Input
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 1024

# Training
BATCH_SIZE = 4
EPOCHS = 10
VAL_INTERVAL = 2
LEARNING_RATE = 0.01
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 0
PREFETCH_FACTOR = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("Device:", DEVICE)


Device: cuda


# Dataset Processing

In [ ]:
ID_TO_TRAIN_ID = np.full(256, IGNORE_INDEX, dtype=np.uint8)

for original_id, train_id in {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4,
    17: 5, 19: 6, 20: 7, 21: 8, 22: 9,
    23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18
}.items():
    ID_TO_TRAIN_ID[original_id] = train_id

def build_cache(split):
    """Resize + remap Avoid decode on each Epoch"""

    image_dir = Path(CACHE_ROOT) / split / "images"
    mask_dir = Path(CACHE_ROOT) / split / "masks"

    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    source = Cityscapes(
        root=DATA_ROOT,
        split=split,
        mode="fine",
        target_type="semantic"
    )

    for index in tqdm(range(len(source)), desc=f"Caching {split}"):

        image_path = image_dir / f"{index:05d}.jpg"
        mask_path = mask_dir / f"{index:05d}.png"

        if image_path.exists() and mask_path.exists():
            continue

        image, mask = source[index]

        image.resize(
            (IMAGE_WIDTH, IMAGE_HEIGHT),
            Image.BILINEAR
        ).save(image_path, quality=95)

        mask = mask.resize(
            (IMAGE_WIDTH, IMAGE_HEIGHT),
            Image.NEAREST
        )

        Image.fromarray(
            ID_TO_TRAIN_ID[np.array(mask, dtype=np.uint8)]
        ).save(mask_path)


build_cache("train")
build_cache("val")


Caching train:   0%|          | 0/2975 [00:00<?, ?it/s]

Caching val: 100%|██████████| 500/500 [00:00<00:00, 28515.22it/s]


In [ ]:
CLASS_NAMES = [
    "road",
    "sidewalk",
    "building",
    "wall",
    "fence",
    "pole",
    "traffic light",
    "traffic sign",
    "vegetation",
    "terrain",
    "sky",
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "train",
    "motorcycle",
    "bicycle"
]


class CityscapesDataset(torch.utils.data.Dataset):

    def __init__(self, split="train", augment=False):

        self.image_dir = Path(CACHE_ROOT) / split / "images"
        self.mask_dir = Path(CACHE_ROOT) / split / "masks"

        self.names = sorted(
            path.stem for path in self.image_dir.glob("*.jpg")
        )

        if not self.names:
            raise RuntimeError(
                f"Empty cache for split '{split}'. Kindly run build cache firstly('{split}')."
            )

        self.augment = augment

        self.color_jitter = RandomApply(
            [
                ColorJitter(
                    brightness=0.2,
                    contrast=0.2,
                    saturation=0.2
                )
            ],
            p=0.5
        )

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):

        name = self.names[index]

        image = Image.open(
            self.image_dir / f"{name}.jpg"
        ).convert("RGB")

        mask = Image.open(
            self.mask_dir / f"{name}.png"
        )

        if self.augment:

            if torch.rand(1).item() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            image = self.color_jitter(image)

        # ImageNet normalization
        image = TF.to_tensor(image)

        # image = TF.normalize(
        #     image,
        #     mean=[0.485, 0.456, 0.406],
        #     std=[0.229, 0.224, 0.225]
        # )

        # Giữ uint8 để tiết kiệm RAM worker; chuyển sang int64 sau khi lên GPU
        mask = torch.from_numpy(np.array(mask, dtype=np.uint8))

        return image, mask


# PidNET

In [ ]:
!git clone https://github.com/XuJiacong/PIDNet.git

In [ ]:
import sys
sys.path.append('./PIDNet')
import models.pidnet as pidnet

class PIDNetWrapper(nn.Module):
    def __init__(self, num_classes=19, arch='pidnet-s'):
        super().__init__()
        
        self.model = pidnet.get_pred_model(arch, num_classes)
        
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        target_h = int(x.shape[-2])
        target_w = int(x.shape[-1])
        
        x = (x - self.mean) / self.std
        
        outputs = self.model(x)
        
        if isinstance(outputs, (tuple, list)):
            out = outputs[0]
        else:
            out = outputs
            
        out = F.interpolate(out, size=(target_h, target_w), mode='bilinear', align_corners=False)
        
        return out

In [ ]:
# Initial model PIDNet
model = PIDNetWrapper(
    num_classes=NUM_CLASSES, 
    arch='pidnet-s'
).to(DEVICE, memory_format=torch.channels_last)

model.eval()

dummy = torch.randn(
    2, 3, IMAGE_HEIGHT, IMAGE_WIDTH
).to(DEVICE, memory_format=torch.channels_last)

with torch.no_grad():
    output = model(dummy)

print(output.shape)


torch.Size([2, 19, 512, 1024])


# Training and Evaluation

## Loss Function

In [82]:
criterion = nn.CrossEntropyLoss(
    ignore_index=IGNORE_INDEX
)

# Confusion Matrix

In [83]:
class SegmentationMetrics:

    def __init__(self, num_classes, device):

        self.num_classes = num_classes

        self.confusion_matrix = torch.zeros(
            (num_classes, num_classes),
            dtype=torch.int64,
            device=device
        )

    @torch.no_grad()
    def update(self, predictions, targets):

        predictions = predictions.argmax(dim=1)

        valid = targets != IGNORE_INDEX

        targets = targets[valid].long()
        predictions = predictions[valid].long()

        indices = (
            targets * self.num_classes + predictions
        )

        matrix = torch.bincount(
            indices,
            minlength=self.num_classes ** 2
        )

        self.confusion_matrix += matrix.reshape(
            self.num_classes,
            self.num_classes
        )

    def compute(self):

        cm = self.confusion_matrix.float()

        tp = torch.diag(cm)

        fp = cm.sum(dim=0) - tp

        fn = cm.sum(dim=1) - tp

        # IoU
        union = tp + fp + fn

        iou = tp / union.clamp(min=1e-8)

        valid_iou = union > 0

        miou = iou[valid_iou].mean()

        # Precision
        precision = tp / (tp + fp).clamp(min=1e-8)

        # Recall
        recall = tp / (tp + fn).clamp(min=1e-8)

        # F1-score
        f1 = (
            2 * precision * recall /
            (precision + recall).clamp(min=1e-8)
        )

        # Pixel accuracy
        pixel_acc = (
            tp.sum() / cm.sum().clamp(min=1e-8)
        )

        return {
            "pixel_accuracy": pixel_acc.item(),
            "mIoU": miou.item(),
            "precision": precision.mean().item(),
            "recall": recall.mean().item(),
            "f1_score": f1.mean().item(),
            "class_iou": iou.cpu().numpy()
        }

## Training

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device
):
    model.train()

    total_loss = torch.zeros((), device=device)
    correct_pixels = torch.zeros((), device=device)
    total_pixels = torch.zeros((), device=device)
    num_samples = 0

    progress = tqdm(loader, desc="Training")

    for step, (images, masks) in enumerate(progress):

        images = images.to(
            device,
            non_blocking=True
        ).contiguous(memory_format=torch.channels_last)

        masks = masks.to(
            device,
            dtype=torch.long,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.detach() * images.size(0)
        num_samples += images.size(0)

        with torch.no_grad():
            predictions = outputs.detach().argmax(dim=1)
            valid = masks != IGNORE_INDEX
            correct_pixels += ((predictions == masks) & valid).sum()
            total_pixels += valid.sum()

        if step % 50 == 0:
            progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = (total_loss / num_samples).item()
    accuracy = (correct_pixels.float() / total_pixels.clamp(min=1)).item()

    return avg_loss, accuracy

In [85]:
def calculate_metrics(confusion_matrix):

    cm = confusion_matrix.float()

    # True Positive
    TP = torch.diag(cm)

    # False Positive
    FP = cm.sum(dim=0) - TP

    # False Negative
    FN = cm.sum(dim=1) - TP

    # IoU
    iou = TP / (TP + FP + FN).clamp(min=1e-7)

    # Precision
    precision = TP / (TP + FP).clamp(min=1e-7)

    # Recall
    recall = TP / (TP + FN).clamp(min=1e-7)

    # F1-score
    f1 = (
        2 * precision * recall
        / (precision + recall).clamp(min=1e-7)
    )

    # Pixel Accuracy
    pixel_accuracy = (
        TP.sum() / cm.sum().clamp(min=1e-7)
    )

    # Mean IoU
    present_classes = (TP + FP + FN) > 0

    miou = iou[present_classes].mean()

    # Macro Precision / Recall / F1
    precision_mean = precision[present_classes].mean()
    recall_mean = recall[present_classes].mean()
    f1_mean = f1[present_classes].mean()

    return {
        "pixel_accuracy": pixel_accuracy.item(),
        "mIoU": miou.item(),
        "precision": precision_mean.item(),
        "recall": recall_mean.item(),
        "f1_score": f1_mean.item(),
        "class_iou": iou.cpu().numpy()
    }

## Validation

In [ ]:
def validate(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    num_samples = 0

    confusion_matrix = torch.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        device=device,
        dtype=torch.int64
    )

    progress = tqdm(loader, desc="Validation")

    with torch.no_grad():
        for images, masks in progress:
            images = images.to(
                device,
                non_blocking=True
            ).contiguous(memory_format=torch.channels_last)

            masks = masks.to(
                device,
                dtype=torch.long,
                non_blocking=True
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, masks)

            total_loss += loss.item() * images.size(0)
            num_samples += images.size(0)

            predictions = outputs.argmax(dim=1)
            valid = masks != IGNORE_INDEX
            
            target_pixels = masks[valid]
            predicted_pixels = predictions[valid]

            indices = target_pixels * NUM_CLASSES + predicted_pixels

            confusion_matrix += torch.bincount(
                indices,
                minlength=NUM_CLASSES * NUM_CLASSES
            ).reshape(NUM_CLASSES, NUM_CLASSES)

    avg_loss = total_loss / num_samples
    metrics = calculate_metrics(confusion_matrix)

    return avg_loss, metrics

## Main Training

In [ ]:
def main():
    # Dataset
    train_dataset = CityscapesDataset(
        split="train",
        augment=True
    )

    val_dataset = CityscapesDataset(
        split="val",
        augment=False
    )

    # DataLoader
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    # Model
    model = PIDNetWrapper(
        num_classes=NUM_CLASSES, 
        arch='pidnet-s'
    ).to(DEVICE, memory_format=torch.channels_last)

    # Loss
    criterion = nn.CrossEntropyLoss(
        ignore_index=IGNORE_INDEX
    )

    optimizer = torch.optim.SGD(
            model.parameters(),
            lr=LEARNING_RATE,
            momentum=0.9,
            weight_decay=WEIGHT_DECAY
        )

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4
    )

    # Mixed precision
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    os.makedirs(
        "checkpoints",
        exist_ok=True
    )

    checkpoint_path = "checkpoints/latest_model_PidNet.pth"
    start_epoch = 0
    best_miou = 0.0

    if os.path.exists(checkpoint_path):
        print(f"Loading checkpoint from {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        if "scheduler_state_dict" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        if "scaler_state_dict" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])
            
        start_epoch = checkpoint["epoch"]
        best_miou = checkpoint.get("best_miou", 0.0)
        
        print(f"Resumed successfully from epoch {start_epoch} | Best mIoU: {best_miou:.4f}")
        
    TOTAL_EPOCHS = 150
    for epoch in range(start_epoch, TOTAL_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{TOTAL_EPOCHS}")
        
        # Training
        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler,
            DEVICE
        )

        # Validation
        val_loss, val_metrics = validate(
            model,
            val_loader,
            criterion,
            DEVICE
        )
        scheduler.step(
            val_metrics["mIoU"]
        )
        current_lr = optimizer.param_groups[0]["lr"]

        print("\nTraining Results")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Pixel Accuracy: {train_accuracy:.4f}")

        print("\nValidation Results")
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Pixel Accuracy: {val_metrics['pixel_accuracy']:.4f}")
        print(f"mIoU: {val_metrics['mIoU']:.4f}")
        print(f"Precision: {val_metrics['precision']:.4f}")
        print(f"Recall: {val_metrics['recall']:.4f}")
        print(f"F1-score: {val_metrics['f1_score']:.4f}")
        
        print(f"Learning Rate: {current_lr:.6f}")

        # Save best model
        if val_metrics["mIoU"] > best_miou:
            best_miou = val_metrics["mIoU"]
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(), 
                    "scaler_state_dict": scaler.state_dict(),       
                    "best_miou": best_miou,
                    "class_names": CLASS_NAMES
                },
                "checkpoints/best_model_PidNet.pth"
            )
            print("Best model saved!")

        # Save latest checkpoint
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(), 
                "scaler_state_dict": scaler.state_dict(),       
                "best_miou": best_miou
            },
            "checkpoints/latest_model_PidNet.pth"
        )

    print("\nTraining completed!")
    print(f"Best mIoU: {best_miou:.4f}")

if __name__ == "__main__":
    main()

Loading checkpoint from checkpoints/latest_model_PidNet.pth...
Resumed successfully from epoch 50 | Best mIoU: 0.5375

Epoch 51/150


Validation: 100%|██████████| 125/125 [00:10<00:00, 12.32it/s]



Training Results
Train Loss: 0.1126
Train Pixel Accuracy: 0.9613

Validation Results
Val Loss: 0.2781
Pixel Accuracy: 0.9190
mIoU: 0.5288
Precision: 0.7497
Recall: 0.6066
F1-score: 0.6541
Learning Rate: 0.001250

Epoch 52/150


Validation: 100%|██████████| 125/125 [00:08<00:00, 14.22it/s]



Training Results
Train Loss: 0.1110
Train Pixel Accuracy: 0.9618

Validation Results
Val Loss: 0.2785
Pixel Accuracy: 0.9187
mIoU: 0.5264
Precision: 0.7455
Recall: 0.6058
F1-score: 0.6521
Learning Rate: 0.001250

Epoch 53/150


Validation: 100%|██████████| 125/125 [00:08<00:00, 14.23it/s]


... (191 outputs left out)

Validation: 100%|██████████| 125/125 [00:27<00:00,  4.59it/s]



Training Results
Train Loss: 0.1019
Train Pixel Accuracy: 0.9647

Validation Results
Val Loss: 0.2843
Pixel Accuracy: 0.9185
mIoU: 0.5205
Precision: 0.7563
Recall: 0.5943
F1-score: 0.6457
Learning Rate: 0.000000

Epoch 150/150


Validation: 100%|██████████| 125/125 [00:28<00:00,  4.44it/s]



Training Results
Train Loss: 0.1020
Train Pixel Accuracy: 0.9647

Validation Results
Val Loss: 0.2857
Pixel Accuracy: 0.9187
mIoU: 0.5246
Precision: 0.7438
Recall: 0.6040
F1-score: 0.6502
Learning Rate: 0.000000

Training completed!
Best mIoU: 0.5375


# Inference – Semantic Segmentation

In [ ]:
def load_model(checkpoint_path, device):

    model = PIDNetWrapper(
        num_classes=NUM_CLASSES, 
        arch='pidnet-s'
    ).to(DEVICE, memory_format=torch.channels_last)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    return model


@torch.no_grad()
def predict(model, image_path, device):

    image = Image.open(image_path).convert("RGB")

    original_size = image.size

    image = TF.resize(
        image,
        [IMAGE_HEIGHT, IMAGE_WIDTH],
        interpolation=InterpolationMode.BILINEAR
    )

    image = TF.to_tensor(image)

    # image = TF.normalize(
    #     image,
    #     mean=[0.485, 0.456, 0.406],
    #     std=[0.229, 0.224, 0.225]
    # )

    image = image.unsqueeze(0).to(device)

    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):

        output = model(image)

    prediction = output.argmax(dim=1)

    prediction = prediction.squeeze(0).cpu().numpy()

    prediction = Image.fromarray(
        prediction.astype(np.uint8)
    )

    prediction = prediction.resize(
        original_size,
        resample=Image.Resampling.NEAREST
    )

    return prediction

In [89]:
CITYSCAPES_COLORS = [
    (128, 64, 128),   # Road
    (244, 35, 232),   # Sidewalk
    (70, 70, 70),     # Building
    (102, 102, 156),  # Wall
    (190, 153, 153),  # Fence
    (153, 153, 153),  # Pole
    (250, 170, 30),   # Traffic light
    (220, 220, 0),    # Traffic sign
    (107, 142, 35),   # Vegetation
    (152, 251, 152),  # Terrain
    (70, 130, 180),   # Sky
    (220, 20, 60),    # Person
    (255, 0, 0),      # Rider
    (0, 0, 142),      # Car
    (0, 0, 70),       # Truck
    (0, 60, 100),     # Bus
    (0, 80, 100),     # Train
    (0, 0, 230),      # Motorcycle
    (119, 11, 32)     # Bicycle
]


def colorize_mask(mask):

    mask = np.array(mask)

    color_mask = np.zeros(
        (*mask.shape, 3),
        dtype=np.uint8
    )

    for class_id, color in enumerate(CITYSCAPES_COLORS):

        color_mask[mask == class_id] = color

    return Image.fromarray(color_mask)

In [90]:
model = load_model(
    "checkpoints/best_model_PidNet.pth",
    DEVICE
)

image_path = "C:\\Users\\TDA5HC\\Desktop\\Documents\\01_CS_Master\\12_DeepLearningAndApplicationInComputerVision\\DeepLearningAndComputerVision\\src\\A2\\data\\leftImg8bit\\val\\frankfurt\\frankfurt_000000_000576_leftImg8bit.png"

prediction = predict(
    model,
    image_path,
    DEVICE
)

color_mask = colorize_mask(prediction)

color_mask.save("segmentation_result.png")